# CEGMem on Colab

Runs the whole study — candidate selection, the oracle gate, the π̂ screen, the
corpus freeze, the E1–E8 grid and the analysis — against a local
`qwen2.5-coder:7b` served by Ollama on the Colab GPU, or against an OpenAI chat
model on `--backend cloud`.

`RUNBOOK.md` in the repository is the authority on what each stage does and why.
This notebook is only the Colab wrapper around it.

**What v3 changes from v2.** The structure is the same — same sections, same
order, same launch-then-status idiom — so what you already know how to drive
still drives. Four things are new:

1. **`RUN_DIR`** in the configuration cell. One variable decides where every
   artifact of this run is written (`data/<RUN_DIR>/…`, `logs/<RUN_DIR>/…`).
   Leave it empty and everything lands in `data/` and `logs/` exactly as v2 did.
   Every stage prints the directory it resolved to as its first line.
2. **The slow-task filter** at the oracle gate (`--reference-timeout`, 10 s). A
   coding task whose correct solution cannot answer one of its own shipped
   inputs in that long is dropped before the expensive half of the gate.
3. **The corpus pin.** `select_corpus.py` now takes the previous corpus first
   within each band, so a re-screen cannot evict a task whose episodes are
   already paid for — and it errors out rather than losing one silently.
4. **Every code cell carries a header** saying which stage it is, what it reads,
   what it writes, how long it takes and whether it can be skipped.

**Three things Colab changes, and they are not cosmetic.**

1. **The session dies.** Free-tier runtimes stop after a few hours, and idle tabs
   stop sooner; the full grid is days of wall clock. So `cache/`, `data/` and
   `logs/` live on Google Drive, and the work is cut into shards small enough to
   finish inside one session. Nothing is lost to a disconnect — one
   `RoundRecord` is written per round as it goes — but a shard that *finishes*
   leaves cleaner books than one that is interrupted.
2. **The GPU decides the context window.** Ollama picks the window from available
   VRAM and **truncates** an over-long prompt instead of refusing it, and that
   window reaches neither the response cache key nor any logged row. A T4 has
   less VRAM than a workstation, so the risk of silently getting 4096 instead of
   32768 is higher here. §11 verifies it before anything is spent; if the check
   fails, stop.
3. **Do not commit from Colab.** `data/` becomes a symlink into Drive, so
   `git status` will report the tracked `data/mutants.py` as deleted. Colab is a
   compute node; results travel back through Drive and commits are made on your
   own machine.

**A cell that blocks is a cell you cannot use.** Every long stage below is
launched into the background and returns immediately. `!tail -f` is not used
anywhere: it never returns, and a Colab tab holding a `tail -f` is a tab you
have to interrupt before you can look at anything else. `scripts/fleet.sh status`
replaces it — re-run that cell as often as you like.

**This screens the gated pool, not the full candidate list.** The gate has
already thrown out everything that cannot become a corpus task, and π̂ for a
rejected candidate is read by nothing — not `select_corpus.py`, which only
looks up pool members, and not `screening.json`, which is built in that same
loop. The last screen spent 8,440 of its 21,040 calls (40%) on tasks the gate
had already rejected. Pass `--pool` to screen a different list.


## 1. Configuration

Edit this cell, then run everything below in order.


In [43]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   configuration - nothing runs
# READS   nothing
# WRITES  python variables only, and RUN_DIR into the environment
# TIME    instant
# SKIP?   NO. Every cell below reads these
# ────────────────────────────────────────────────────────────────────────────
# ── repository ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/dat95201/ceg-mem.git"   # private repo: https://<TOKEN>@github.com/...
BRANCH   = "official/2026-09-01"
WORKDIR  = "/content/ceg-mem"

# ── which run this is ───────────────────────────────────────────────────────
# One variable decides where every artifact of this run is written:
#
#     RUN_DIR = "official-2026-09-01"   ->  data/official-2026-09-01/...
#                                           logs/official-2026-09-01/...
#     RUN_DIR = ""                      ->  data/ and logs/, exactly as before
#
# It reaches every stage through the environment - src/paths.py reads it for
# python, scripts/run_dir_paths.sh for bash - so nothing below has to pass it
# along. Every stage prints the directory it resolved to as its first line.
#
# Set it to keep this run's output apart from the corpus and episodes already
# on Drive. Leave it empty to write into the shared data/ the way v2 did.
RUN_DIR = "official-2026-09-01"

import os
os.environ["RUN_DIR"] = RUN_DIR        # inherited by every ! cell below

# ── where results persist across sessions ───────────────────────────────────
RUNS = "/content/drive/MyDrive/ceg-mem-runs"

# ── the ConDefects test data (several GB) ───────────────────────────────────
# Preferred: a path inside your mounted Drive. Leave as None to fall back to the
# file id below, which downloads over the network instead.
TEST_ZIP_DRIVE_PATH = None      # e.g. "/content/drive/MyDrive/ceg-mem/Test.zip"
TEST_ZIP_FILE_ID    = "https://drive.google.com/file/d/1avUulLRNNVoWLStKKiqJyckwUux1ghzd/view?usp=sharing"   # bare id or a full Drive URL

# ── the protocol. These are not preferences: every one of them changes what the
#    measured quantities ARE, and a shard measured under a different value is a
#    different instrument. Keep them identical across every machine and session
#    for the life of the study. See RUNBOOK.md §1.
MODEL          = "qwen2.5-coder:7b"
CONTEXT_LENGTH = 32768
TEMPERATURE    = 1.0
OLLAMA_PORT    = 11435          # our own port, not a desktop app's

# ── how many shards a fleet cuts a range into ───────────────────────────────
# Ollama runs with OLLAMA_NUM_PARALLEL=1, so shards do NOT get parallel
# generation - their model calls queue at the server. What overlaps is the
# oracle: each candidate patch runs in a sandbox subprocess, which is CPU, off
# the GPU's critical path. So the useful count is bounded by cores, not VRAM,
# and the gain flattens fast. 4-6 is the honest range on a T4.
SHARDS = 6

# ── the corpus gates (RUNBOOK.md section 2, scripts/validate_oracle.py) ──────
# REFERENCE_TIMEOUT is the SLOW-TASK FILTER. A coding task whose *correct*
# solution cannot answer one of its own shipped inputs within this many seconds
# is dropped at the gate, before stage 2's mutant judging. It is a property of
# the coding task - reference solution plus the problem's own test data - and
# is measured on the run criterion 2 was already making, so it costs nothing
# and references no arm's results.
#
# Measured over all 526 Stage-0 candidates:
#     30 s (the sandbox timeout)   drops 0 of the 106-task corpus
#     10 s                         drops 3 of 106, 11 of the 315-task pool,
#                                  34 of 526 candidates      <- the default
#      5 s                         drops 7 of 106
# There is no cliff in the distribution, so this is a budget dial, not a
# discovery. Raise it to keep more tasks and pay for them.
REFERENCE_TIMEOUT = 10.0

# No band below this many tasks. The primary comparison is per band, so a band
# of four cannot carry the claim it is there to test. 5 bands x 10 = a corpus
# of at least 50.
MIN_PER_BAND = 10

# The gate's cohort. "auto" = every eligible fault; a number = a declared
# cohort the walk must reach exactly, or the pool will not freeze. Leave it on
# auto unless you are reproducing a specific earlier freeze.
CORPUS_SIZE = "auto"

# ── the second proposer (optional, §17). Leave the key empty to stay local. ──
# Only ever run on the 30-task sweep subset, never on the reported grid: pi is a
# property of the model, the corpus was banded under qwen, and `model` is in the
# cell key so the two can never pool. See RUNBOOK.md and PLAN §2.4.
CLOUD_MODEL   = "gpt-4o-mini"
CLOUD_API_KEY = ""              # sk-... ; empty means "do not run the cloud arm"
CLOUD_PRICE_IN, CLOUD_PRICE_OUT = 0.15, 0.60    # USD per Mtok, gpt-4o-mini
CLOUD_CONTEXT = 128000
CLOUD_BUDGET  = 5.0             # TOTAL for the fleet; fleet.sh divides it by SHARDS

## 2. Check the runtime has a GPU


In [44]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   check the runtime has a GPU
# READS   nvidia-smi
# WRITES  nothing
# TIME    instant
# SKIP?   no - a CPU runtime makes the model stages ~20x slower
# ────────────────────────────────────────────────────────────────────────────
!nvidia-smi -L || echo "NO GPU — Runtime > Change runtime type > T4 GPU, then rerun"

/bin/bash: line 1: nvidia-smi: command not found
NO GPU — Runtime > Change runtime type > T4 GPU, then rerun


A 7B model at Q4_K_M is about 4.7 GB and fits a T4 alongside a 32k window. On CPU
each call takes minutes rather than seconds, and the grid never finishes — so
this is a hard prerequisite, not a nicety.

## 3. Mount Drive and create the persistent directories


In [45]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   mount Drive, create the run directories
# READS   your Drive
# WRITES  {RUNS}/cache, {RUNS}/data, {RUNS}/logs
# TIME    ~10 s
# SKIP?   NO. Without it nothing survives the session
# ────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
for sub in ("cache", "data", "logs"):
    os.makedirs(f"{RUNS}/{sub}", exist_ok=True)
print("persistent root:", RUNS)
!ls -la {RUNS}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
persistent root: /content/drive/MyDrive/ceg-mem-runs
total 122
drwx------ 2 root root   4096 Aug 20 07:34 cache
-rw------- 1 root root 111656 Aug 29 09:29 CEGMem_Colab.ipynb
drwx------ 6 root root   4096 Aug 29 10:19 data
drwx------ 3 root root   4096 Aug 29 10:19 logs


## 4. Clone the repository


In [46]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   clone the repository
# READS   GitHub
# WRITES  /content/ceg-mem
# TIME    ~20 s
# SKIP?   no
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir('/content')
!test -d {WORKDIR} || git clone {REPO_URL} {WORKDIR}
import os; os.chdir(WORKDIR)
!git fetch --all --quiet && git checkout {BRANCH} && git pull --ff-only
!git log --oneline -1

D	cache/.gitkeep
D	data/.gitkeep
D	data/calls_screen.jsonl
D	data/calls_screen_021_030.jsonl
D	data/calls_screen_031_040.jsonl
D	data/calls_screen_041_050.jsonl
D	data/calls_screen_051_060.jsonl
D	data/calls_screen_061_070.jsonl
D	data/calls_screen_071_080.jsonl
D	data/calls_screen_081_090.jsonl
D	data/calls_screen_091_100.jsonl
D	data/calls_screen_101_110.jsonl
D	data/calls_screen_111_120.jsonl
D	data/calls_screen_121_130.jsonl
D	data/calls_screen_131_140.jsonl
D	data/calls_screen_135_140.jsonl
D	data/calls_screen_141_150.jsonl
D	data/calls_screen_147_150.jsonl
D	data/calls_screen_151_200.jsonl
D	data/calls_screen_172_180.jsonl
D	data/calls_screen_181_190.jsonl
D	data/calls_screen_184_190.jsonl
D	data/calls_screen_191_200.jsonl
D	data/calls_screen_197_200.jsonl
D	data/calls_screen_201_225.jsonl
D	data/calls_screen_226_250.jsonl
D	data/calls_screen_251_263.jsonl
D	data/calls_screen_254_263.jsonl
D	data/calls_screen_264_274.jsonl
D	data/calls_screen_275_294.jsonl
D	data/calls_screen_293

## 5. Point `cache/`, `data/` and `logs/` at Drive

This is the cell that makes a multi-session run possible. `data/mutants.py` is
**source, not a result** — it is copied into Drive first so the symlink does not
hide it.

In [47]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   point cache/ data/ logs/ at Drive
# READS   {RUNS}
# WRITES  three symlinks in the checkout
# TIME    instant
# SKIP?   NO. This is what makes a multi-session run possible
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# mutants.py is tracked source; it must exist inside the Drive-backed data dir
!cp -n data/mutants.py {RUNS}/data/ 2>/dev/null; true

!rm -rf cache data logs
!ln -s {RUNS}/cache cache
!ln -s {RUNS}/data  data
!ln -s {RUNS}/logs  logs

!ls -la | grep -E ' (cache|data|logs)'
print("--- data/ must contain mutants.py ---")
!ls data/

lrwxrwxrwx  1 root root    41 Aug 29 11:53 cache -> /content/drive/MyDrive/ceg-mem-runs/cache
lrwxrwxrwx  1 root root    40 Aug 29 11:53 data -> /content/drive/MyDrive/ceg-mem-runs/data
lrwxrwxrwx  1 root root    40 Aug 29 11:53 logs -> /content/drive/MyDrive/ceg-mem-runs/logs
--- data/ must contain mutants.py ---
calls_eval_E1_001_018.jsonl
calls_eval_E1_019_036.jsonl
calls_eval_E1_037_054.jsonl
calls_eval_E1_055_072.jsonl
calls_eval_E1_073_089.jsonl
calls_eval_E1_090_106.jsonl
calls_eval_E2_001_018.jsonl
calls_eval_E2_019_036.jsonl
calls_eval_E2_037_054.jsonl
calls_eval_E2_055_072.jsonl
calls_eval_E2_073_089.jsonl
calls_eval_E2_090_106.jsonl
calls_eval_E3-steer-only_001_018.jsonl
calls_eval_E3-steer-only_019_036.jsonl
calls_eval_E3-steer-only_037_054.jsonl
calls_eval_E3-steer-only_055_072.jsonl
calls_eval_E3-steer-only_073_089.jsonl
calls_eval_E3-steer-only_090_106.jsonl
calls_eval_E4-k20_001_005.jsonl
calls_eval_E4-k20_006_010.jsonl
calls_eval_E4-k20_011_015.jsonl
calls_eval_E4-k20_

### Where is this run writing?

`RUN_DIR` nests **inside** the Drive-backed `data/`, so a named run persists
across sessions exactly like the default one — `data` is already a symlink into
Drive, and `data/official-2026-09-01/` is a directory under it.

Run this before anything else. A run that quietly wrote to the wrong directory
is discovered here, in one line, instead of at merge time.

In [ ]:
import os; os.chdir(WORKDIR)
os.environ["RUN_DIR"] = RUN_DIR         # again: this cell may be re-run alone

!python3 src/paths.py
!bash -c '. scripts/run_dir_paths.sh; echo "bash agrees: $RUN_DATA  $RUN_LOGS"'

# Both halves must name the same directory. tests/test_run_dir.py asserts it,
# but the cheapest place to notice a typo in RUN_DIR is here.
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
os.makedirs(RUN_DATA, exist_ok=True); os.makedirs(RUN_LOGS, exist_ok=True)
print("\nthis session writes to:", RUN_DATA, "and", RUN_LOGS)

### Seed the run from what has already been paid for

`RUN_DIR` gives this run its own directory, and every stage looks for its inputs
*there* — so a fresh run directory would re-screen 526 candidates from scratch,
which is hours of model calls that have already been bought.

Two files are copied in, and only two:

| file | why it is reused |
|---|---|
| `candidates.json` | Stage 0's seeded order. Deterministic from the same ConDefects tree, so re-deriving it would produce the identical file. |
| `screen_merged.json` | the π̂ screen. **This is the expensive one** — hours of model calls, and π̂ is a property of the task and the model, neither of which this run changes. |

Everything else is re-derived, because the point of this run is that it changes:
the gate re-runs under the slow-task filter, the corpus is re-selected against
the pin, and the grid is re-measured under the fixed guard.

`tasks.json` is deliberately **not** copied. It stays where it is and acts as
the *pin* — `select_corpus.py` reads the previous corpus from the base `data/`
so the new selection cannot evict a task whose episodes are already paid for.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   seed the run directory from the base run
# READS   data/candidates.json, data/screen_merged.json
# WRITES  data/<RUN_DIR>/candidates.json, data/<RUN_DIR>/screen_merged.json
# TIME    ~5 s
# SKIP?   only if RUN_DIR is empty (then there is nothing to seed)
# ────────────────────────────────────────────────────────────────────────────
import os, shutil; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"

REUSE = ["candidates.json", "screen_merged.json"]

if not RUN_DIR:
    print("RUN_DIR is empty - this run writes to data/ directly, nothing to seed")
else:
    os.makedirs(RUN_DATA, exist_ok=True)
    for name in REUSE:
        src, dst = f"data/{name}", f"{RUN_DATA}/{name}"
        if os.path.exists(dst):
            print(f"  keep    {dst}  (already seeded)")
        elif os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"  copied  {src}  ->  {dst}")
        else:
            print(f"  MISSING {src} - the stage that builds it has to run in this RUN_DIR")

# The pin is read from the base path, not copied. Say so out loud: if this file
# is gone, the re-selection has no cache-reuse anchor and select_corpus.py will
# happily freeze a corpus that evicts tasks you have already paid for.
print()
print("pin:", "data/tasks.json",
      "present" if os.path.exists("data/tasks.json") else "MISSING - no cache-reuse anchor")

> The response cache is thousands of small files and Drive is slow with those. If
> a stage crawls, keep `cache/` on local disk instead and copy it to Drive after
> each stage (`!rsync -a cache/ {RUNS}/cache/`). The trade is that calls bought
> since the last sync are lost when the session ends — they are re-bought, not
> corrupted.

## 6. Python dependencies


In [27]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   python dependencies
# READS   requirements.txt
# WRITES  site-packages
# TIME    ~1 min
# SKIP?   yes, on a warm runtime - but it costs seconds to confirm
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!pip install -q -r requirements.txt
!apt-get -qq install -y lsof > /dev/null
!python3 -c "import openai, dotenv, numpy, scipy, matplotlib; print('deps ok')"

deps ok


## 7. Clone ConDefects (code only)


In [28]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   clone ConDefects (code only, no test data)
# READS   GitHub
# WRITES  external/ConDefects
# TIME    ~30 s
# SKIP?   yes if external/ConDefects already exists
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!python3 scripts/fetch_condefects.py

external/ConDefects already present - skipping clone
external/ConDefects/Test already unpacked - leaving it alone

root:               external/ConDefects
faulty programs:    2864 across 985 coding tasks
with test data:     411 programs across 106 coding tasks
contest dates:      2021-10-02 .. 2024-06-30

ready. Next: python3 scripts/validate_oracle.py


## 8. The contest test data

`fetch_condefects.py` can only clone the *code*. The test data ships separately,
and without it there are no inputs, so there is no oracle and nothing below runs.

Put the archive at `external/ConDefects/Test.zip` and let the script unpack it —
it also handles the mirrors that wrap the tree one level deeper (`Test/Test/…`)
and verifies the layout afterwards.

**A partial archive is fine, and is the normal case here.** The full test tree
unpacks to roughly 63 GB; an archive holding only the coding tasks in
`data/candidates.json` is a few GB. Either works, and no cell changes between
them, as long as the archive's entries are rooted at `Test/<contest>/<LETTER>/`
— unpacking into `external/ConDefects/` then produces exactly the layout
`src.adapter.test_dir_for` looks under. The name does not matter: the cell below
copies whatever you point it at to `external/ConDefects/Test.zip`, which is where
`fetch_condefects.py` expects to find it.

**One stage does need the whole tree, and it is easy to miss.** Stage 0 gates on
the shipped test data — `G1_no_expected_output` requires a coding task to ship at
least one expected output, and `G4` counts test cases — so running
`pipeline.sh candidates` against a partial archive silently produces a
*different* candidate list: thousands of faults fail G1 for want of data rather
than for anything about the fault. With a partial archive, bring
`data/candidates.json` over as an artifact instead of regenerating it. The stage
cell below checks for this and refuses rather than quietly renumbering every
shard.

Everything downstream is safe: the gate, the screen, the corpus freeze and the
grid only ever touch tasks in the candidate list, and the gate's sibling faults
are other submissions to the *same* coding task, so they share its test
directory.

**Disk.** `Test/` must stay on Colab's local disk — never symlink it into Drive,
which has nowhere near the room.

In [29]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   fetch the contest test data (several GB)
# READS   your Drive or a Drive file id
# WRITES  external/ConDefects/Test.zip
# TIME    5-25 min, once per runtime
# SKIP?   yes if Test/ is already unpacked
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
import os, re, shutil, zipfile

dst = "external/ConDefects/Test.zip"

# A Drive download that hits the virus-scan interstitial or a quota wall saves
# the HTML error page under the target name. Catch that here, loudly: left alone
# it surfaces much later as a BadZipFile, or worse as a stage that finds no data.
def check(path):
    size = os.path.getsize(path)
    if not zipfile.is_zipfile(path):
        head = open(path, "rb").read(160)
        print(f"{path} is {size/1e6:.1f} MB and is NOT a zip archive.")
        print("It is Drive's HTML error page saved under the archive's name.")
        print("first bytes:", head)
        print()
        print("Fix by either:")
        print("  - setting TEST_ZIP_DRIVE_PATH to the file inside your mounted Drive")
        print("    (no quota, no confirm token - this is the reliable path), or")
        print("  - sharing the file 'Anyone with the link' to allow an anonymous download.")
        raise SystemExit(1)
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
    print(f"{path}: {size/2**30:.2f} GB, {len(names):,} entries, root {names[0]!r}")

if os.path.exists("external/ConDefects/Test"):
    print("Test/ already unpacked - skipping download")
elif TEST_ZIP_DRIVE_PATH and os.path.exists(TEST_ZIP_DRIVE_PATH):
    print("copying from the mounted Drive ...")
    shutil.copyfile(TEST_ZIP_DRIVE_PATH, dst)
    check(dst)
else:
    # gdown wants a bare file id. Handed a /file/d/<id>/view?usp=... URL it
    # downloads the VIEW PAGE - a few dozen KB of HTML under the archive's name,
    # which is the failure this extraction exists to prevent. Accept either form.
    m = (re.search(r"/d/([A-Za-z0-9_-]{20,})", TEST_ZIP_FILE_ID)
         or re.search(r"[?&]id=([A-Za-z0-9_-]{20,})", TEST_ZIP_FILE_ID))
    FILE_ID = m.group(1) if m else TEST_ZIP_FILE_ID.strip()
    print("downloading file id:", FILE_ID)
    !pip install -q gdown
    !gdown {FILE_ID} -O {dst}
    check(dst)

!df -h /content | tail -1

Test/ already unpacked - skipping download
overlay         226G   33G  193G  15% /


In [30]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   unpack the test data
# READS   Test.zip
# WRITES  external/ConDefects/Test/
# TIME    ~3 min
# SKIP?   yes if Test/ is already unpacked
# ────────────────────────────────────────────────────────────────────────────
!unzip external/ConDefects/Test.zip -d external/ConDefects

Archive:  external/ConDefects/Test.zip
replace external/ConDefects/Test/agc065/C/out/02_many_random_case_06.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

### Does the archive cover what the study needs?

A partial archive is only safe if it covers every task the pipeline will walk.
This is the cheapest place to find out — the alternative is a stage failing hours
in, or worse, a screen quietly reporting `pi_hat = 0.0` for a task whose oracle
had nothing to test against.

In [31]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   does the archive cover what the study needs?
# READS   external/ConDefects/Test/
# WRITES  nothing
# TIME    ~10 s
# SKIP?   no - a partial tree silently changes which faults exist
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
import sys, json; sys.path.insert(0, ".")
from src.adapter import discover, test_dir_for, TEST_DIR

TASKS = discover()
task_ids  = {t.task_id for t in TASKS.values()}
with_data = {tid for tid in task_ids if test_dir_for(tid) is not None}
print(f"coding tasks in Code/      : {len(task_ids)}")
print(f"of those, with test data   : {len(with_data)}")
print(f"archive looks              : {'PARTIAL' if len(with_data) < 0.9*len(task_ids) else 'complete'}")

# If the candidate list is already here, check the archive covers all of it.
if os.path.exists(f"{RUN_DATA}/candidates.json"):
    names = [c["name"] for c in json.load(open(f"{RUN_DATA}/candidates.json"))["candidates"]]
    missing = [n for n in names if n.split("/")[0] not in with_data]
    print(f"\ncandidates                 : {len(names)}")
    print(f"without test data          : {len(missing)}  {missing[:5]}")
    print("OK - the archive covers every candidate" if not missing
          else "STOP - those candidates cannot be screened or run")
else:
    print("\nno {RUN_DATA}/candidates.json yet - see the candidates stage below")

coding tasks in Code/      : 985
of those, with test data   : 106
archive looks              : PARTIAL

candidates                 : 526
without test data          : 420  ['abc320_f/47539324', 'arc156_c/38986255', 'arc162_e/43020372', 'abc354_g/53647241', 'abc293_f/45665153']
STOP - those candidates cannot be screened or run


## 9. Ollama, with the context window pinned


In [32]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   install Ollama
# READS   ollama.com
# WRITES  /usr/local/bin/ollama
# TIME    ~1 min
# SKIP?   yes on a runtime that already has it
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# The installer unpacks a zstd-compressed bundle and Colab images do not ship
# zstd, so it fails with "This version requires zstd for extraction".
!apt-get -qq install -y zstd curl > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [33]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   serve the model with the context window PINNED
# READS   the ollama binary
# WRITES  a server on 127.0.0.1:$OLLAMA_PORT
# TIME    ~15 s
# SKIP?   NO. This is where the 32k window is set
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
import subprocess, os, time, urllib.request

# subprocess.Popen, not `!ollama serve &`: a background job started from a `!`
# cell can die with the subshell that launched it.
os.makedirs("logs", exist_ok=True)
env = dict(os.environ,
           OLLAMA_HOST=f"127.0.0.1:{OLLAMA_PORT}",
           OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH),
           OLLAMA_NUM_PARALLEL="1",
           OLLAMA_KEEP_ALIVE="60m")
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("logs/ollama.log", "a"), stderr=subprocess.STDOUT)

for _ in range(40):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2)
        print("server up on 127.0.0.1:%d" % OLLAMA_PORT); break
    except Exception:
        time.sleep(1)
else:
    print("server did not come up - see logs/ollama.log")

server up on 127.0.0.1:11435


In [34]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   pull the model weights
# READS   ollama registry
# WRITES  ~4.7 GB in the ollama store
# TIME    3-8 min, once per runtime
# SKIP?   yes if the weights are already pulled
# ────────────────────────────────────────────────────────────────────────────
!OLLAMA_HOST=127.0.0.1:{OLLAMA_PORT} ollama pull {MODEL}

## 10. `.env`

`.env` is gitignored, so it does not travel with the repository. This is the
client half of the protocol; `screen_shard.sh` and `eval_shard.sh` export their
own values over it, so a drifted `.env` cannot corrupt a shard — but the
interactive tools read it.

In [35]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   write .env - the protocol, in one file
# READS   the config cell
# WRITES  .env
# TIME    instant
# SKIP?   NO. src/llm.py reads this
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# REASONING_EFFORT is new on this branch: src/llm.py routes o-series models
# differently (max_completion_tokens, no temperature) and refuses the pairing of
# a chat model with a non-empty effort rather than silently splitting the cells.
# Empty is correct for qwen and for gpt-4o-mini.
env_text = f"""LLM_BASE_URL=http://127.0.0.1:{OLLAMA_PORT}/v1
LLM_API_KEY=unused
MODEL={MODEL}
TEMPERATURE={TEMPERATURE}
REASONING_EFFORT=
LLM_TIMEOUT_SEC=1800
LLM_MAX_RETRIES=5
LLM_CONTEXT_TOKENS={CONTEXT_LENGTH}
PRICE_IN_PER_MTOK=0
PRICE_OUT_PER_MTOK=0
BUDGET_USD_CAP=1.0
SANDBOX_TIMEOUT_SEC=30.0
SANDBOX_RECURSION_LIMIT=10000
CONDEFECTS_ROOT=external/ConDefects
# CACHE_DIR is NOT under RUN_DIR on purpose: the response cache is what
# makes a re-run free, and scoping it per run would cold-start every sweep.
# CALLS_LOG is left EMPTY so the ledger follows RUN_DIR; every shard
# overrides it with its own file anyway.
CACHE_DIR=cache
CALLS_LOG=
"""
open(".env", "w").write(env_text)
print(env_text)

LLM_BASE_URL=http://127.0.0.1:11435/v1
LLM_API_KEY=unused
MODEL=qwen2.5-coder:7b
TEMPERATURE=1.0
REASONING_EFFORT=
LLM_TIMEOUT_SEC=1800
LLM_MAX_RETRIES=5
LLM_CONTEXT_TOKENS=32768
PRICE_IN_PER_MTOK=0
PRICE_OUT_PER_MTOK=0
BUDGET_USD_CAP=1.0
SANDBOX_TIMEOUT_SEC=30.0
SANDBOX_RECURSION_LIMIT=10000
CONDEFECTS_ROOT=external/ConDefects
CACHE_DIR=cache
CALLS_LOG=data/calls.jsonl



## 11. Verify before spending anything

`serve_local.sh` loads the model, asks `/api/ps` what is **actually** being
served, and refuses on a mismatch. `context_length` must read 32768.

If it reads 4096, stop here. The prompt would be silently cropped, and it would
be cropped worst on the arms that carry the most accumulated evidence — which are
exactly the arms the experiment compares.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   verify before spending anything
# READS   .env, the served model
# WRITES  nothing
# TIME    ~30 s
# SKIP?   NO. This is the cell that catches a 4096-token window before it costs you days
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/serve_local.sh --port {OLLAMA_PORT}
!bash scripts/pipeline.sh

---

## 12. What this branch adds, and why it changes the run order

If you last ran this notebook for the screen, everything below §12 is different.
`git log` and `RUNBOOK.md` are the record; this is the short list of what shows
up as new *cells*.

| | |
|---|---|
| **a 4th arm** | `transcript` — the ChatRepair baseline. The `untyped` arm shows the proposer nothing, which a reviewer reads as a straw man; this is the condition real agents actually implement. `--exp E6-transcript`. |
| **2 more `c` levels** | `E5-c25`, `E5-c00`. Four points gave a slope but no crossover; `c*` needs to see typed fall below untyped. |
| **a redundancy audit** | `--exp E8-audit` pays the oracle on guarded rounds so their failure type is on the record. Without it every θ-based redundancy count is censored exactly where an arm guards. Sweep subset only — it spends the time E2 exists to show can be saved. |
| **a regression audit** | `--check-regression` scores an accepted patch on **both** halves of the shipped pool: the cases the faulty version fails (what the loop optimises) and the cases it passes (which nothing in the loop ever checks). Not in the cell key, so it can be switched on without invalidating finished cells. |
| **a cloud backend** | `--backend cloud` on both shard scripts. Same wire format, different base URL; o-series routing included. §17. |
| **3 post-hoc scripts** | `measure_redundancy.py`, `measure_patch_quality.py`, `measure_typing_coherence.py`. All `$0`, all read logs that already exist. |
| **`scripts/fleet.sh`** | N background shards of one stage, one manifest, one status command. This is what makes the cells below non-blocking. |

**The one ordering rule that costs real money to get wrong: run `E1` first.**
`E2`'s `untyped` cell and `E3-guard-only` build a byte-identical prompt to
`E1`'s, so they replay E1's cached draws for free. Running them first buys the
same draw twice.

### `fleet.sh` in one cell

```
bash scripts/fleet.sh screen --shards 5 --from 401 --to 450 --calls 40
bash scripts/fleet.sh eval   --exp E1 --shards 6
bash scripts/fleet.sh eval   --exp E2 --shards 6 -- --check-regression
bash scripts/fleet.sh status      # non-blocking, re-run freely
bash scripts/fleet.sh tail 3      # BLOCKS (it is tail -f). Ctrl-C to stop.
bash scripts/fleet.sh wait        # BLOCKS until drained, then unloads the model
bash scripts/fleet.sh stop
```

Everything after a bare `--` is forwarded verbatim to the shard script.

Three things it does that hand-typed `nohup` lines do not:

* **Ranges** are contiguous, non-overlapping and cover the range exactly, with
  the remainder at the *front* so no shard is left running alone at the end.
* **`--no-stop-model` on every shard.** Both shard scripts unload the weights on
  exit *by default and on purpose*, including when they are only borrowing a
  server. So the first of six shards to finish runs `ollama stop` on the model
  the other five are still calling, and they each pay a cold reload.
  `--keep-serving` does **not** prevent this — it keeps the server process, not
  the resident model. `fleet.sh wait` does the single unload at the end.
* **On `--backend cloud` it divides `BUDGET_USD_CAP`** by the shard count.
  `src.llm.spent()` reads only its own process's ledger and every shard has one,
  so six shards each honouring a $25 cap is a real ceiling of $150.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   check out the branch and prove the corpus is visible
# READS   GitHub, data/<RUN_DIR>
# WRITES  nothing
# TIME    ~15 s
# SKIP?   no
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# Pull the branch, then prove fleet.sh is there and sees the frozen corpus.
# The three numbers it prints are read off data/eval_order.txt and
# data/sweep_programs.txt, not written down in the script - if they are "-" the
# corpus has not been frozen in this Drive yet.
!git fetch --all --quiet && git checkout {BRANCH} && git pull --ff-only
!git log --oneline -1
!bash scripts/eval_shard.sh -h | sed -n '/--from N --to M/,/writes it from/p'

---

# The pipeline

Run these in order. `pipeline.sh` refuses to start a stage whose input artifact
is missing, so a skipped step is caught rather than silently producing a smaller
result.

**Sizing a shard to a session.** At roughly 20 s per call, a session safely holds
**1,500–2,000 calls**. Cut smaller than feels necessary: an interrupted shard
loses nothing, but a finished one leaves cleaner books.

## Stage: candidates — no model calls, a few minutes

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   candidates - Stage 0, the screening pool. No model calls
# READS   external/ConDefects/
# WRITES  data/<RUN_DIR>/candidates.json
# TIME    ~3 min
# SKIP?   yes if candidates.json exists for this RUN_DIR
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
import sys, json, hashlib; sys.path.insert(0, ".")

# The candidate LIST reproduces on a partial tree: dedup() draws once per coding
# task over the faults that pass, and every task that survives to G6 has its test
# data here, so the RNG stream is identical. What does NOT reproduce is
# control_unfiltered (drawn over G1-only survivors, a population this tree does
# not contain) and funnel (attribution shifts to G1). Take those two from a
# full-tree run; this file's copies of them are not usable.
EXPECTED_ORDER_SHA = "ac1bd4508978fc80798487a534fdb7b43deff2a17438aa3b33f73c97062db21d"

!bash scripts/pipeline.sh candidates

d = json.load(open(f"{RUN_DATA}/candidates.json"))
names = [c["name"] for c in d["candidates"]]
digest = hashlib.sha256("\n".join(names).encode()).hexdigest()

print("candidates      :", d["n_candidates"])
print("distinct tasks  :", len({n.split('/')[0] for n in names}))
print("order sha256    :", digest)
print("matches full-tree run:", digest == EXPECTED_ORDER_SHA)
print()
print("funnel (NOT comparable to a full-tree run):", json.dumps(d["funnel"]))
print("control arm     :", len(d["control_unfiltered"]), "tasks - drawn over this tree only, do not report")

This decides which faults the study is ever allowed to see, and its output order
is the traversal every later index range is cut from. Run it **once** and treat
it as frozen — re-running it with different flags renumbers every shard.

## Stage: the oracle gate — no model calls, ~2 h

Launched in the background. The second cell is the progress check — re-run it as
often as you like; it returns immediately. This is the pattern used for every
long stage below.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   gate - E0, the oracle gate + the SLOW-TASK FILTER. No model calls
# READS   data/<RUN_DIR>/candidates.json
# WRITES  data/<RUN_DIR>/pool/tasks.json, pool/oracle_validation.json
# TIME    ~2 h at --jobs 6; 6-12 h at --jobs 1 (a publication freeze)
# SKIP?   yes if the pool is already frozen for this RUN_DIR - it refuses to overwrite without --force
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
os.makedirs(RUN_LOGS, exist_ok=True)
# --corpus-size auto, not a number. The gate refuses to freeze unless the
# cohort reaches exactly --corpus-size, and the surviving count moves the
# moment REFERENCE_TIMEOUT does - the previous freeze used 315, after the
# filter it is ~304, and there is no way to know that before the walk.
# `auto` makes the cohort every eligible fault, which is also the one
# denominator that cannot be tuned to flatter the pass rate.
!nohup bash scripts/oracle_gate.sh --jobs 1 \
    --reference-timeout {REFERENCE_TIMEOUT} \
    --corpus-size {CORPUS_SIZE} \
    > {RUN_LOGS}/gate.log 2>&1 &
print("launched; check with the next cell")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   watch the gate
# READS   logs/<RUN_DIR>/gate.log
# WRITES  nothing
# TIME    instant
# SKIP?   re-run as often as you like
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
!pgrep -fa validate_oracle.py | head -2 || echo "not running"
!grep -cE '  (PASS|SKIP|SLOW|INEL)  ' {RUN_LOGS}/gate.log || true
# SLOW is the slow-task filter; SKIP is the gate proper. Two different reasons.
!grep -c '  SLOW  ' {RUN_LOGS}/gate.log || echo '0 slow so far'
!tail -4 {RUN_LOGS}/gate.log

### What the slow-task filter dropped

`SLOW` in the log is a **cost** verdict, not a correctness one: the coding
task's own correct solution could not answer one of its own shipped inputs
within `REFERENCE_TIMEOUT`. Those tasks never reach stage 2's mutant judging,
which is where the gate's time goes.

The measured seconds are recorded on every record — passing ones too — so the
threshold can be re-evaluated from the frozen report without re-running the
gate.

In [ ]:
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
import json, pathlib
p = pathlib.Path(f"{RUN_DATA}/pool/oracle_validation.json")
if not p.exists():
    print("the gate has not finished yet")
else:
    r = json.loads(p.read_text())
    print(f"examined {r['n_examined']}  usable {r['n_usable']}  "
          f"slow {r['n_slow']}  (threshold {r['reference_timeout_sec']}s)")
    for n in r["slow_tasks"][:20]:
        print("   SLOW", n)
    # Re-derive the verdict at another threshold, from this file alone.
    secs = [(f.get("reference_sec_max") or 0, n) for n, f in r["faults"].items()]
    for T in (5, 10, 15, 20, 30):
        print(f"   at {T:2d}s the filter would drop "
              f"{sum(1 for s, _ in secs if s >= T)} of {len(secs)}")

## Stage: the π̂ screen — a fleet, uses the model

Shards are contiguous index ranges over `data/candidates.json` and **must not
overlap** — `fleet.sh` is what guarantees that. Any contiguous range is already
balanced, so a shard is a smaller screen rather than a skewed one.

`--calls 40` is not optional in practice: π̂ lives on a grid of `1/K`, and at
`K = 10` nothing can land in `hard` = [0.02, 0.08) at all — the band the paper
predicts its largest effect in. `fleet.sh` warns if you omit it and names the
depth `data/screen_merged.json` was measured at.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   screen - E0b, measure pi_hat. A fleet, uses the model
# READS   data/<RUN_DIR>/candidates.json
# WRITES  data/<RUN_DIR>/screen_<range>.json, calls_screen_*.jsonl
# TIME    hours; cut into SHARDS and resumable
# SKIP?   yes if screen_merged.json exists for this RUN_DIR
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
FROM, TO = 1, 526
!bash scripts/fleet.sh screen --shards {SHARDS} --from {FROM} --to {TO} --calls 40 --port {OLLAMA_PORT}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   fleet status
# READS   logs/<RUN_DIR>/fleet/
# WRITES  nothing
# TIME    instant
# SKIP?   re-run as often as you like
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   merge the screen shards
# READS   data/<RUN_DIR>/screen_*.json
# WRITES  data/<RUN_DIR>/screen_merged.json
# TIME    ~10 s
# SKIP?   no - the corpus freeze needs it
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# Coverage is NOT proven by a drained fleet - that only says the processes
# exited. This is the audit.
!python3 scripts/consolidate_screens.py

## Stage: freeze the corpus — no model calls

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   corpus - freeze the corpus on the pi bands. No model calls
# READS   data/<RUN_DIR>/pool/tasks.json, screen_merged.json, and the PIN (data/tasks.json)
# WRITES  data/<RUN_DIR>/tasks.json, screening.json
# TIME    ~5 s
# SKIP?   yes if tasks.json is frozen for this RUN_DIR
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
# --pin defaults to the BASE corpus (plain data/tasks.json), so the tasks whose
# episodes are already paid for are taken first within their band and cannot be
# evicted by a re-screen. select_corpus.py errors out if any of them is lost -
# read what it names before reaching for --allow-unpinned.
!python3 scripts/select_corpus.py \
    --pool {RUN_DATA}/pool/tasks.json \
    --screen {RUN_DATA}/screen_merged.json \
    --min-calls 38 \
    --min-per-band {MIN_PER_BAND}

import json, collections
d = json.load(open(f"{RUN_DATA}/tasks.json"))
print()
print(d["n_selected"], "tasks;", dict(collections.Counter(t["stratum"] for t in d["tasks"])))
print("screen depth K =", d["selection"]["min_calls"])

pin = d["selection"].get("pin", {})
print(f"pin: {pin.get('n_kept')} / {pin.get('n_pinned')} of the paid corpus kept")
for row in pin.get("lost", []):
    print("   LOST", row["name"], "-", row["why"])
print("   (a kept task replays its cached episodes even if it changed band:")
print("    band is not in src.loop.cell_signature)")

> **What the pin will and will not stop for.** A pinned task the slow-task
> filter dropped, or that the gate rejected, is *reported and the run
> continues* — it is gone on its merits and no quota brings it back. A
> pinned task evicted because its **band filled up** is an error, because
> nothing is wrong with it and raising that band's quota gets it back with
> its episodes intact. `--allow-unpinned` downgrades only the second kind.
>
> Expect three settled losses on this run — `abc303_g`, `abc325_f`,
> `abc356_c`, all dropped by the 10 s filter. The corpus still comes out at
> 106: their bands have surplus in the pool, so three fresh tasks take their
> places. 103 tasks keep their paid episodes; the 3 new ones run cold.


> **Read that output before going further.** π̂ lives on a grid of `1/K`, so a
> band can only be filled if some multiple of `1/K` falls inside it. At `K = 10`,
> for example, nothing can land in `hard` = [0.02, 0.08) at all — and that is the
> band where the paper predicts its largest effect.
>
> An empty primary band is fixed by deepening the screen, **not** by proceeding.
> It has to happen now: re-freezing the corpus once episodes exist invalidates
> them, and `eval_shard.sh` refuses to cut a shard from a corpus whose digest no
> longer matches the order it was built from.

## Stage: the oracle's blind spot — no model calls

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   pool-strength - the oracle's blind spot. No model calls
# READS   data/<RUN_DIR>/tasks.json
# WRITES  data/<RUN_DIR>/pool_strength.json
# TIME    ~20 min at --jobs 4
# SKIP?   yes - it is a robustness check, not an input to the grid
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh pool-strength --jobs 4

---

## Stage: the grid — fleets, uses the model

### First the trial, then a smoke test of every preset that has never run

The trial is three tasks, one seed, B=5, on a log that can never be merged into
reported data. It is not a smoke test of the model, it is a test of the flags and
of the server.

Then the same for the four presets this branch adds. The check that matters is
the **second** run of each: every cell must print `already complete, skipping`,
in seconds. Anything else means the resume key does not match the index — and a
multi-day grid is the wrong place to discover that. This branch put
`audit_guarded`, `transcript_window` and `reasoning_effort` into the cell key, so
this is exactly the run that would catch a mistake in that.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   re-fetch ConDefects after a runtime recycle
# READS   GitHub
# WRITES  external/ConDefects
# TIME    ~30 s
# SKIP?   yes unless the runtime was recycled
# ────────────────────────────────────────────────────────────────────────────
!python3 scripts/fetch_condefects.py

external/ConDefects already present - skipping clone
external/ConDefects/Test already unpacked - leaving it alone

root:               external/ConDefects
faulty programs:    2864 across 985 coding tasks
with test data:     411 programs across 106 coding tasks
contest dates:      2021-10-02 .. 2024-06-30

ready. Next: python3 scripts/validate_oracle.py


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   check the ConDefects tree without re-fetching
# READS   external/ConDefects
# WRITES  nothing
# TIME    ~5 s
# SKIP?   yes
# ────────────────────────────────────────────────────────────────────────────
!python3 scripts/fetch_condefects.py --check-only


root:               external/ConDefects
faulty programs:    2864 across 985 coding tasks
with test data:     411 programs across 106 coding tasks
contest dates:      2021-10-02 .. 2024-06-30

ready. Next: python3 scripts/validate_oracle.py


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   smoke test: the trial universe, 3 tasks x 3 arms
# READS   data/<RUN_DIR>/trial_programs.txt
# WRITES  data/<RUN_DIR>/episodes_trial.jsonl
# TIME    ~5 min
# SKIP?   no - this is the cheapest place to find a broken flag
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
!bash scripts/pipeline.sh eval --exp trial --port {OLLAMA_PORT}
!python3 scripts/summarize.py --episodes-path {RUN_DATA}/episodes_trial.jsonl

+ bash scripts/eval_shard.sh --exp trial --port 11435
3 programs -> data/eval_shards/trial_001_003.txt

experiment   trial
shard        1-3 of 3 (3 tasks, universe data/trial_programs.txt)
grid         modes="no_memory untyped typed"  seeds="1"  budget=5
             9 cells  ·  --check-overfit
backend      ollama
protocol     model=qwen2.5-coder:7b  temperature=1.0  context=32768
             granularity=fine  sandbox_timeout=30.0
episodes     data/episodes_trial.jsonl   (trial - never merged)
ledger       data/calls_trial.jsonl
resume       none (--no-resume-merged, or no merged history yet)

server already on 127.0.0.1:11435 - adopting it
qwen2.5-coder:7b already present - nothing to download
loading qwen2.5-coder:7b and verifying the served window
  ok: {"backend": "ollama", "context_length": 32768, "model_digest": "dae161e27b0e90dd", "quantization": "Q4_K_M"}

ready. .env already points at http://127.0.0.1:11435 with MODEL=qwen2.5-coder:7b.
next: RUNBOOK.md - the run order, or bas

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   smoke test: one cell per preset, budget 5
# READS   data/<RUN_DIR>/eval_order.txt
# WRITES  data/<RUN_DIR>/episodes_eval_*.jsonl
# TIME    ~10 min
# SKIP?   yes, but a preset that fails here fails 6 hours in otherwise
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
for exp in ["E3-guard-only", "E3-steer-only", "E4-k3", "E5-c50",
            "E5-c25", "E5-c00", "E5-random", "E6-transcript", "E8-audit"]:
    print(f"\n===== {exp} =====")
    !bash scripts/pipeline.sh eval --exp {exp} --from 1 --to 1 --seeds 1 --budget 5 --port {OLLAMA_PORT}


===== E3-guard-only =====
+ bash scripts/eval_shard.sh --exp E3-guard-only --from 1 --to 1 --seeds 1 --budget 5 --port 11435
1 programs -> data/eval_shards/E3-guard-only_001_001.txt

experiment   E3-guard-only
shard        1-1 of 106 (1 tasks, universe data/eval_order.txt)
grid         modes="typed"  seeds="1"  budget=5
             1 cells  ·  --steer off
backend      ollama
protocol     model=qwen2.5-coder:7b  temperature=1.0  context=32768
             granularity=fine  sandbox_timeout=30.0
episodes     data/episodes_eval_E3-guard-only_001_001.jsonl
ledger       data/calls_eval_E3-guard-only_001_001.jsonl
resume       none (--no-resume-merged, or no merged history yet)

server already on 127.0.0.1:11435 - adopting it
qwen2.5-coder:7b already present - nothing to download
loading qwen2.5-coder:7b and verifying the served window
  ok: {"backend": "ollama", "context_length": 32768, "model_digest": "dae161e27b0e90dd", "quantization": "Q4_K_M"}

ready. .env already points at http://127.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   smoke test: the same, with the tail of each log
# READS   as above
# WRITES  as above
# TIME    ~10 min
# SKIP?   yes
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# Run it again. Every line must say "already complete, skipping".
for exp in ["E5-c25", "E5-c00", "E5-random", "E6-transcript", "E8-audit"]:
    print(f"\n===== {exp} (second run) =====")
    !bash scripts/pipeline.sh eval --exp {exp} --from 1 --to 1 --seeds 1 --budget 5 --port {OLLAMA_PORT} 2>&1 | tail -3


===== E5-c25 (second run) =====
total spent so far: $0.0000

unloaded qwen2.5-coder:7b

===== E5-c00 (second run) =====
total spent so far: $0.0000

unloaded qwen2.5-coder:7b

===== E5-random (second run) =====
total spent so far: $0.0000

unloaded qwen2.5-coder:7b

===== E6-transcript (second run) =====
total spent so far: $0.0000

unloaded qwen2.5-coder:7b

===== E8-audit (second run) =====
total spent so far: $0.0000

unloaded qwen2.5-coder:7b


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   delete the smoke-test episodes
# READS   nothing
# WRITES  removes the trial logs
# TIME    instant
# SKIP?   NO if you ran the smoke tests - trial rows must not reach the grid
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
# Clear the rehearsal. The cached completions stay and E1 replays them free.
!rm -f {RUN_DATA}/episodes_trial.jsonl {RUN_DATA}/overfit_trial.jsonl {RUN_DATA}/calls_trial.jsonl

### What each arm costs, before you start it

Run this cell before the grid and again whenever you re-freeze anything. It is
the answer to "how long will this take", derived from **your** corpus rather
than from a number somebody typed once: `E[rounds]` comes from the π̂ these 106
tasks were banded on, and the latency from your own call ledgers.

Read the `free arms` line. Three of the arms build a prompt byte-identical to
one already bought, so they cost **zero model calls** — that is what the
common-random-numbers design buys, and it is also the falsifier: those arms must
agree with E1 about `success@B` exactly, or the guard is unsound.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   what each arm costs, before you start it
# READS   data/<RUN_DIR>/tasks.json
# WRITES  nothing
# TIME    ~5 s
# SKIP?   yes, but read it once
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
# ── the run plan, computed from YOUR corpus, not written down ────────────────
# Every number below is derived: E[rounds] from the pi_hat this corpus was
# actually banded on, latency from your own call ledgers. Re-freeze the corpus
# or screen deeper and this cell tells you the new answer instead of the old
# one. That is the same discipline fleet.sh applies to universe sizes.
import json, glob, statistics as st, pathlib

B = 20
tasks = json.load(open(f"{RUN_DATA}/tasks.json"))["tasks"]
screen = json.load(open(f"{RUN_DATA}/screen_merged.json"))["per_program"]
sweep = [l.strip() for l in open(f"{RUN_DATA}/sweep_programs.txt")
         if l.strip() and not l.startswith("#")]
NC, NS = len(tasks), len(sweep)

def e_rounds(pi):
    """E[min(first accept, B)] - a memory arm stops at its first accept, so the
    band structure, not the task count, decides what an arm costs."""
    return float(B) if pi <= 0 else (1 - (1 - pi) ** B) / pi

pi = {t["name"]: screen[t["name"]]["pi_hat"] for t in tasks}
er = {n: e_rounds(p) for n, p in pi.items()}
ER_C, ER_S = sum(er.values()), sum(er[n] for n in sweep)

secs = []
for p in glob.glob(f"{RUN_DATA}/calls_*.jsonl"):
    for line in open(p):
        try: d = json.loads(line)
        except Exception: continue
        if isinstance(d.get("sec"), (int, float)): secs.append(d["sec"])
LAT_MED = st.median(secs) if secs else 20.0
LAT_MEAN = st.mean(secs) if secs else 34.0

print(f"corpus {NC} tasks | sweep {NS} | latency measured on {len(secs):,} calls: "
      f"median {LAT_MED:.1f}s, mean {LAT_MEAN:.1f}s\n")
print(f"{'band':10s} {'n':>3s} {'pi mean':>8s} {'E[rounds]':>10s} {'share':>7s}")
for b in ("dead", "hard", "medium", "easy", "too_easy"):
    ns = [t["name"] for t in tasks if t["stratum"] == b]
    if not ns: continue
    s = sum(er[n] for n in ns)
    print(f"{b:10s} {len(ns):3d} {st.mean(pi[n] for n in ns):8.4f} "
          f"{st.mean(er[n] for n in ns):10.2f} {s/ER_C*100:6.1f}%")

# (exp, universe rounds, n modes, seeds, pays_for_calls)
# pays_for_calls=False means the arm builds a prompt byte-identical to one
# already bought, so every draw is a cache hit. That is the CRN design, and it
# is also the guard-soundness falsifier: those arms MUST agree with E1 about
# success@B, exactly.
PLAN = [
    ("E1",             NC * B, 1, 5, True,  "--force-full-budget: 20 rounds, no early stop"),
    ("E2 untyped",     ER_C,   1, 5, False, "prompt identical to E1 -> cache"),
    ("E2 typed",       ER_C,   1, 5, True,  "memory in the prompt"),
    ("E3-guard-only",  ER_C,   1, 3, False, "steer off -> E1's prompt -> cache"),
    ("E3-steer-only",  ER_C,   1, 3, True,  "typed prompt, guard off"),
    ("E4-k20/k8/k3",   ER_S,   3, 3, True,  "upper bound: a weaker oracle accepts sooner"),
    ("E5 x5 +random",  ER_S,   6, 3, True,  "6 levels on the sweep"),
    ("E6-transcript",  ER_C,   1, 5, True,  "LOW end; see the note below"),
    ("E8-audit",       ER_S,   2, 3, False, "same draws as E2 -> cache; oracle time only"),
]
print(f"\n{'experiment':16s} {'cells':>6s} {'rounds':>8s} {'new calls':>10s} "
      f"{'GPU-h':>7s} {'4h sess':>8s}  why")
print("-" * 96)
tot_cells = tot_rounds = tot_new = 0
for name, rounds_u, modes, seeds, pays, why in PLAN:
    n_universe = NC if rounds_u == ER_C else (NS if rounds_u == ER_S else NC)
    cells = n_universe * modes * seeds
    rounds = rounds_u * modes * seeds
    new = rounds if pays else 0
    hours = new * LAT_MEAN / 3600
    print(f"{name:16s} {cells:6d} {rounds:8.0f} {new:10.0f} {hours:7.1f} "
          f"{hours/4:8.1f}  {why}")
    tot_cells += cells; tot_rounds += rounds; tot_new += new
print("-" * 96)
print(f"{'TOTAL':16s} {tot_cells:6d} {tot_rounds:8.0f} {tot_new:10.0f} "
      f"{tot_new*LAT_MEAN/3600:7.1f} {tot_new*LAT_MEAN/3600/4:8.1f}")
lo, hi = tot_new * LAT_MED / 3600, tot_new * LAT_MEAN / 3600
print(f"\nGPU time: {lo:.0f}-{hi:.0f} hours = {lo/24:.1f}-{hi/24:.1f} days of T4.")
print(f"Free arms: {tot_rounds - tot_new:,.0f} of {tot_rounds:,.0f} rounds cost NO model call.")
print(f"\nE6-transcript is a RANGE, not a number. The low figure above assumes it")
print(f"behaves like no-memory. The earlier pilot had the guard block 16-19 of 20")
print(f"rounds, which would put it at full budget = {NC*B*5:,} calls "
      f"(+{(NC*B*5-ER_C*5)*LAT_MEAN/3600:.0f} GPU-h). Plan for the high end.")
print(f"\nOLLAMA_NUM_PARALLEL=1, so model calls QUEUE at the server: a fleet does")
print(f"not beat these hours, it keeps the GPU from idling on oracle time.")

corpus 106 tasks | sweep 30 | latency measured on 8 calls: median 5.7s, mean 4.7s

band         n  pi mean  E[rounds]   share
dead        20   0.0000      20.00   37.5%
hard        30   0.0408      14.11   39.6%
medium      20   0.1425       6.86   12.8%
easy        21   0.2667       3.88    7.6%
too_easy    15   0.6117       1.74    2.4%

experiment        cells   rounds  new calls   GPU-h  4h sess  why
------------------------------------------------------------------------------------------------
E1                  530    10600      10600    13.9      3.5  --force-full-budget: 20 rounds, no early stop
E2 untyped          530     5339          0     0.0      0.0  prompt identical to E1 -> cache
E2 typed            530     5339       5339     7.0      1.7  memory in the prompt
E3-guard-only       318     3203          0     0.0      0.0  steer off -> E1's prompt -> cache
E3-steer-only       318     3203       3203     4.2      1.0  typed prompt, guard off
E4-k20/k8/k3        270     

### Where you got to, and how to cut an arm into sessions

15 arms, 3,746 cells, and a runtime that dies every few hours. Run this after
every reconnect: it reads the episode logs on Drive rather than a note, so it is
right even if the last session ended mid-shard or someone else ran a chunk.

`session_split(exp, calls_per_session)` is the part the shard flags do not
cover. `fleet.sh --shards N` splits a range across parallel shards **inside one
runtime**; this splits the arm across **runtimes**, which is the dimension that
kills a multi-day run. It sizes chunks on the same `E[rounds]` as the plan cell,
so a chunk of `dead` tasks comes out smaller than a chunk of `easy` ones — a
flat split by task count would hand one session four times the work of another.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   where you got to, and how to cut an arm into sessions
# READS   data/<RUN_DIR>/episodes*.jsonl
# WRITES  nothing
# TIME    ~20 s
# SKIP?   re-run as often as you like
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
# ── the worklist: what is done, what is left, how to cut it into sessions ────
# A 3-week run across dying Colab runtimes needs one place that answers "where
# was I". This reads the episode logs on Drive - the artifacts, not a note - so
# it is right after a disconnect, and right if someone else ran a shard.
import os, json, glob, collections, pathlib

# In order. E1 first is not a preference: E2's untyped arm and E3-guard-only
# replay E1's cached draws, so running them first buys the same draw twice.
ARMS = ["E1", "E2", "E3-guard-only", "E3-steer-only",
        "E4-k20", "E4-k8", "E4-k3",
        "E5-c90", "E5-c75", "E5-c50", "E5-c25", "E5-c00", "E5-random",
        "E8-audit", "E6-transcript"]

def cells_done(exp):
    """Distinct (task, mode, seed) with at least one round logged, for one exp."""
    seen = set()
    for p in glob.glob(f"{RUN_DATA}/episodes_eval_{exp}_*.jsonl") + [f"{RUN_DATA}/episodes.jsonl"]:
        if not os.path.exists(p):
            continue
        for line in open(p):
            try: r = json.loads(line)
            except Exception: continue
            seen.add((r["task"], r["mode"], r.get("seed", 0)))
    return seen

print(f"{'#':>3s} {'--exp':16s} {'shard logs':>10s} {'cells seen':>11s}  state")
print("-" * 62)
for i, exp in enumerate(ARMS, 1):
    logs = glob.glob(f"{RUN_DATA}/episodes_eval_{exp}_*.jsonl")
    n = len(cells_done(exp)) if logs else 0
    state = "not started" if not logs else ("in progress" if n else "log present, no rounds")
    print(f"{i:3d} {exp:16s} {len(logs):10d} {n:11d}  {state}")
print("\nmerged corpus:", f"{RUN_DATA}/episodes.jsonl present"
      if os.path.exists(f"{RUN_DATA}/episodes.jsonl") else "not merged yet")
print("Coverage is NOT what this shows. Run: bash scripts/pipeline.sh eval --merge --dry-run")


def session_split(exp, calls_per_session=1500, universe=None):
    """--from/--to chunks sized to survive one Colab session.

    A shard is not a session. fleet.sh splits a range across parallel shards
    inside ONE runtime; this splits the arm across runtimes, which is the
    dimension that actually kills a multi-day run. Sized on the SAME E[rounds]
    the plan cell computed, so a chunk of `dead` tasks is smaller than a chunk
    of `easy` ones - a flat split by task count would hand one session four
    times the work of another.
    """
    tasks_j = json.load(open(f"{RUN_DATA}/tasks.json"))["tasks"]
    screen = json.load(open(f"{RUN_DATA}/screen_merged.json"))["per_program"]
    order_file = (f"{RUN_DATA}/sweep_programs.txt"
                  if exp.startswith(("E4", "E5", "E8")) else f"{RUN_DATA}/eval_order.txt")
    order = [l.strip() for l in open(order_file) if l.strip() and not l.startswith("#")]
    B = 20
    full = exp == "E1"
    seeds = 5 if exp in ("E1", "E2", "E6-transcript") else 3
    modes = 2 if exp in ("E2", "E8-audit") else 1
    def cost(name):
        pi = screen[name]["pi_hat"]
        r = B if full else (B if pi <= 0 else (1 - (1 - pi) ** B) / pi)
        return r * seeds * modes
    print(f"\n{exp}: {len(order)} tasks over {order_file}, "
          f"{seeds} seed(s) x {modes} mode(s), ~{calls_per_session} calls/session")
    lo, run, n = 1, 0.0, 0
    for i, name in enumerate(order, 1):
        run += cost(name); n += 1
        if run >= calls_per_session or i == len(order):
            print(f"   --from {lo:3d} --to {i:3d}   {n:3d} tasks, ~{run:5.0f} calls")
            lo, run, n = i + 1, 0.0, 0


# Uncomment for the arm you are about to start. 1500 calls is ~8.6 h at the
# measured mean; drop it to 700 for a 4-hour free-tier session.
# session_split("E1", 1500)
# session_split("E6-transcript", 1500)

  # --exp            shard logs  cells seen  state
--------------------------------------------------------------
  1 E1                        0           0  not started
  2 E2                        0           0  not started
  3 E3-guard-only             1           1  in progress
  4 E3-steer-only             1           1  in progress
  5 E4-k20                    0           0  not started
  6 E4-k8                     0           0  not started
  7 E4-k3                     1           1  in progress
  8 E5-c90                    0           0  not started
  9 E5-c75                    0           0  not started
 10 E5-c50                    1           1  in progress
 11 E5-c25                    1           1  in progress
 12 E5-c00                    1           1  in progress
 13 E5-random                 1           1  in progress
 14 E8-audit                  1           2  in progress
 15 E6-transcript             1           1  in progress

merged corpus: not merged yet


### E1 — run this arm first

`--force-full-budget` is what makes E1 an estimator rather than just a baseline:
the no-memory prompt is byte-identical across all rounds, so every round is an
independent draw of π. It is also the arm every unconditioned arm below replays
its cache from, which is why it goes first.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   eval E1 - no_memory, full budget. Uses the model
# READS   data/<RUN_DIR>/eval_order.txt
# WRITES  data/<RUN_DIR>/episodes_eval_E1_*.jsonl
# TIME    hours; sharded, resumable
# SKIP?   no - E2's baseline and the theory fit both need it
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh eval --exp E1 --shards {SHARDS} --port {OLLAMA_PORT}


03:32:10Z fleet: eval E1  range 1-106 of 106  into 6 shard(s)
  budget    local backend: calls priced at 0, so the cap is a tripwire and division is moot
  server    --keep-serving --no-stop-model on every shard; 'fleet.sh wait' unloads once at the end
  logs      logs/fleet/eval-E1-20260827T033210Z/

  shard 01     1-18    pid 26008    01.log
  shard 02    19-36    pid 26122    02.log
  shard 03    37-54    pid 26204    03.log
  shard 04    55-72    pid 26273    04.log
  shard 05    73-89    pid 26347    05.log
  shard 06    90-106   pid 26423    06.log

launched; this command is done. Next:
  bash scripts/fleet.sh status
  bash scripts/fleet.sh tail 1


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   fleet status
# READS   logs/<RUN_DIR>/fleet/
# WRITES  nothing
# TIME    instant
# SKIP?   re-run as often as you like
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status


  stage=eval
  exp=E1
  from=1
  to=106
  shards=6
  port=11435
  backend=ollama
  forwarded=

  #   RANGE       PID      STATE    ELAPSED   PROGRESS
  -   -----       ---      -----    -------   --------
  1   1-18        26008    done     -         90/90
  2   19-36       26122    done     -         90/90
  3   37-54       26204    done     -         90/90
  4   55-72       26273    done     -         90/90
  5   73-89       26347    done     -         85/85
  6   90-106      26423    done     -         85/85

  0 running · 6 done · 0 failed

  the fleet has drained. Coverage is NOT proven by that - audit it:
    python3 scripts/consolidate_evals.py --dry-run


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   audit coverage of what has merged so far
# READS   data/<RUN_DIR>/episodes*.jsonl
# WRITES  nothing
# TIME    ~30 s
# SKIP?   no - a drained fleet does not mean full coverage
# ────────────────────────────────────────────────────────────────────────────
!python3 scripts/consolidate_evals.py --dry-run

33 shard log(s):
  episodes_eval_E1_001_018.jsonl
  episodes_eval_E1_019_036.jsonl
  episodes_eval_E1_037_054.jsonl
  episodes_eval_E1_055_072.jsonl
  episodes_eval_E1_073_089.jsonl
  episodes_eval_E1_090_106.jsonl
  episodes_eval_E2_001_018.jsonl
  episodes_eval_E2_019_036.jsonl
  episodes_eval_E2_037_054.jsonl
  episodes_eval_E2_055_072.jsonl
  episodes_eval_E2_073_089.jsonl
  episodes_eval_E2_090_106.jsonl
  episodes_eval_E3-guard-only_001_001.jsonl
  episodes_eval_E3-guard-only_001_018.jsonl
  episodes_eval_E3-guard-only_019_036.jsonl
  episodes_eval_E3-guard-only_037_054.jsonl
  episodes_eval_E3-guard-only_055_072.jsonl
  episodes_eval_E3-guard-only_073_089.jsonl
  episodes_eval_E3-guard-only_090_106.jsonl
  episodes_eval_E3-steer-only_001_001.jsonl
  episodes_eval_E3-steer-only_001_018.jsonl
  episodes_eval_E3-steer-only_019_036.jsonl
  episodes_eval_E3-steer-only_037_054.jsonl
  episodes_eval_E3-steer-only_055_072.jsonl
  episodes_eval_E3-steer-only_073_089.jsonl
  episodes_eval

### E9's universe — optional, `fleet.sh` builds it for you

`data/live_programs.txt` is the sweep minus the dead band: the 24 tasks where
π̂ > 0, i.e. the only ones where handing the search more attempts can convert
into a repair. Including the six dead tasks would take E9 from ~13 GPU-h to ~46,
all of it spent re-establishing that dead stays dead.

It is **derived, not drawn**, so `eval_shard.sh` generates it on demand exactly
as it generates `eval_order.txt` and `sweep_programs.txt` — and rebuilds it if
its corpus digest has gone stale. You do not have to run this cell before
`E9-freeguard`; run it only if you want to see the split before committing GPU
hours to it.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   build E9's universe
# READS   data/<RUN_DIR>/sweep_programs.txt
# WRITES  data/<RUN_DIR>/live_programs.txt
# TIME    instant
# SKIP?   yes - fleet.sh builds it on demand
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
!python3 scripts/build_live_universe.py
!head -6 {RUN_DATA}/live_programs.txt

### The remaining arms

Run one fleet at a time — `fleet.sh` refuses to launch a second while shards are
alive rather than interleaving two manifests. Change `EXP` and re-run.

| `--exp` | universe | state | cost |
|---|---|---|---|
| **`E9-freeguard`** | **24 (live)** | **RUN THIS** | draws past `B` are new calls; ~7–14 GPU-h |
| **`E8-corpus`** | **106** | **RUN THIS** | **no model calls**, ~2.3 h on 6 shards |
| `E1` | 106 | done | — |
| `E2` | 106 tasks | done | `untyped` replays E1's cache free; `typed` is new |
| `E3-guard-only` | 106 | done | replays E1's cache free |
| `E3-steer-only` | 106 | done | new calls |
| `E4-k20 k8 k3` | 30 (sweep) | done | new calls; read off `is_truly_correct`, not `accept` |
| `E5-c90 c75 c50 c25 c00` | 30 (sweep) | done, superseded | new calls |
| `E5-random` | 30 (sweep) | done, superseded | the c axis's **null** — classes assigned at random. `c=0.00` is not this |
| `E8-audit` | 30 (sweep) | done | no model calls, oracle time only |

**Everything marked `done` is already in `data/episodes.jsonl` and will print
"already complete, skipping" if you run it.** The preset still works — nothing was
taken away — but re-running buys nothing. The two marked RUN THIS are the only
arms that are work.

**`E9-freeguard` — the new arm.** With a guarded round charged to the attempt
budget the guard cannot change an outcome: it only ever blocks a candidate some
stored counterexample already refutes, so under common random numbers the round
of the first accepted patch is invariant to it, and `untyped` reproduces
`no_memory` to the last decimal at every budget. `--free-guarded-rounds` charges
a blocked proposal one model call and **no** unit of the attempt budget, so the
attempts the guard saves go back to the search. It is the only accounting in
which Corollary 4.4 and Definition 3.1 can come out either way. Draws `1..B`
replay E1's cache free; only draws past `B` are new calls, and only on cells that
did not already accept. Run the cell above this table first — it writes
`data/live_programs.txt`, which E9's universe reads.

**`E8-corpus` — E8-audit over all 106 tasks instead of the 30-task sweep.** It
pays the oracle on guarded rounds so they carry a failure type, which is what
makes FSRR, the type entropy and the revisit curve comparable across arms. The
CRN join recovers this for free wherever the prompt is unconditioned, but it
cannot reach the *steered* typed arm, whose draws diverge. Replays E2's cached
draws: **no model calls.**

**`E5-*` still runs, and you no longer need it.** The knob works — mistyping fires
at 0.825 / 0.633 / 0.432 / 0.181 / 0.077 against a predicted 1−c — but what it
moves is the guard's block rate (0.225 at c=0 to 0.319 at c=1), and the guard
cannot reach the outcome under a charged budget, so success stays flat at
0.71 ± 0.02. `c` is an E9 question. `scripts/measure_typing_coherence.py` measures
the real `c` from the logs with no model calls.

**`E6-transcript` no longer exists.** The preset, the `transcript` mode and the
`--transcript-window` flag were removed from the code; `--exp E6-transcript` now
fails with an unknown-preset error. It tested no surviving claim, and the one
claim it was built for — a typed index staying flat where a transcript grows —
is falsified by the typed arm alone: measured prompt growth is 0.1 tok/round for
`no_memory`, 3.5 for `untyped`, **80.7 for `typed`**.

`-- --check-regression` on E2 turns `accept` into a verdict: it scores the
accepted patch on the cases the faulty version already passed, which nothing in
the loop ever looks at. Sandbox time, no model calls, not in the cell key.

In [36]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   eval - the remaining arms. Uses the model
# READS   data/<RUN_DIR>/eval_order.txt
# WRITES  data/<RUN_DIR>/episodes_eval_<exp>_*.jsonl
# TIME    hours per arm; sharded, resumable
# SKIP?   E2 no; E3-E9 are ablations
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# Change EXP and re-run this cell. One fleet at a time.
#
#   RUN NOW    E9-freeguard   E8-corpus
#   done       E1  E2  E3-guard-only  E3-steer-only
#              E4-k20 E4-k8 E4-k3
#              E5-c90 E5-c75 E5-c50 E5-c25 E5-c00 E5-random   (superseded)
#              E8-audit
#   removed    E6-transcript  -> the preset no longer exists
#
# Anything on the `done` list still works and will print "already complete,
# skipping" for every cell. That is the resume index doing its job, not an error.
EXP = "E9-freeguard"
# EXTRA = ["--check-regression"]        # [] for none
EXTRA = []
extra = " ".join(EXTRA)
!bash scripts/fleet.sh eval --exp {EXP} --shards {SHARDS} --port {OLLAMA_PORT} -- {extra}

fleet: a fleet is still running - 'fleet.sh status', then 'stop' or 'wait'


In [38]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   background status watcher
# READS   logs/<RUN_DIR>/fleet/
# WRITES  logs/<RUN_DIR>/fleet-status.log
# TIME    instant
# SKIP?   yes - convenience only
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
!pkill -f cegmem-status-watch 2>/dev/null; true
!nohup bash -c 'for i in $(seq 1 2880); do { date -u +"status @ %Y-%m-%dT%H:%M:%SZ"; echo; bash scripts/fleet.sh status 2>&1; } > {RUN_LOGS}/.status.tmp; mv -f {RUN_LOGS}/.status.tmp {RUN_LOGS}/status.log; grep -qE "^ *0 running" {RUN_LOGS}/status.log && break; sleep 60; done # cegmem-status-watch' >/dev/null 2>&1 &

^C


In [48]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   fleet status
# READS   logs/<RUN_DIR>/fleet/
# WRITES  nothing
# TIME    instant
# SKIP?   re-run as often as you like
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status


  stage=eval
  exp=E8-audit
  from=1
  to=30
  shards=6
  port=11435
  backend=ollama
  forwarded=

  #   RANGE       PID      STATE    ELAPSED   PROGRESS
  -   -----       ---      -----    -------   --------
  1   1-5         3002     done     -         30/30
  2   6-10        3075     done     -         30/30
  3   11-15       3141     done     -         30/30
  4   16-20       3222     done     -         30/30
  5   21-25       3292     done     -         30/30
  6   26-30       3366     done     -         30/30

  0 running · 6 done · 0 failed

  the fleet has drained. Coverage is NOT proven by that - audit it:
    python3 scripts/consolidate_evals.py --dry-run


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   tail one shard
# READS   logs/<RUN_DIR>/fleet/
# WRITES  nothing
# TIME    instant
# SKIP?   yes
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh tail 2

`fleet.sh wait` is the one blocking cell in this notebook. Run it when you want
the tab to hold until the fleet drains — it also does the single model unload
that every shard was told not to do itself.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   block until the fleet drains
# READS   logs/<RUN_DIR>/fleet/
# WRITES  nothing
# TIME    as long as the fleet runs
# SKIP?   yes - this is the ONE blocking cell in the notebook
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh wait

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   stop the fleet
# READS   nothing
# WRITES  SIGTERMs the shards
# TIME    instant
# SKIP?   yes - shards resume on an identical re-run
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# Only if you need the GPU back or got a range wrong. Shards resume on an
# identical re-launch: every model call already bought replays from the cache
# on Drive, and only the oracle - which is not cached - runs again.
!bash scripts/fleet.sh stop


### Merge, and audit the merge

The audit is the only thing that speaks to coverage. A fleet reporting every
shard `done` says every process exited 0 and nothing more.

In [49]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   audit the merge before doing it
# READS   data/<RUN_DIR>/episodes_eval_*.jsonl
# WRITES  nothing
# TIME    ~1 min
# SKIP?   NO. Read this before the merge, not after
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh eval --merge --dry-run     # coverage, gaps, protocol

+ python3 scripts/consolidate_evals.py --dry-run
93 shard log(s):
  episodes_eval_E1_001_018.jsonl
  episodes_eval_E1_019_036.jsonl
  episodes_eval_E1_037_054.jsonl
  episodes_eval_E1_055_072.jsonl
  episodes_eval_E1_073_089.jsonl
  episodes_eval_E1_090_106.jsonl
  episodes_eval_E2_001_018.jsonl
  episodes_eval_E2_019_036.jsonl
  episodes_eval_E2_037_054.jsonl
  episodes_eval_E2_055_072.jsonl
  episodes_eval_E2_073_089.jsonl
  episodes_eval_E2_090_106.jsonl
  episodes_eval_E3-guard-only_001_001.jsonl
  episodes_eval_E3-guard-only_001_018.jsonl
  episodes_eval_E3-guard-only_019_036.jsonl
  episodes_eval_E3-guard-only_037_054.jsonl
  episodes_eval_E3-guard-only_055_072.jsonl
  episodes_eval_E3-guard-only_073_089.jsonl
  episodes_eval_E3-guard-only_090_106.jsonl
  episodes_eval_E3-steer-only_001_001.jsonl
  episodes_eval_E3-steer-only_001_018.jsonl
  episodes_eval_E3-steer-only_019_036.jsonl
  episodes_eval_E3-steer-only_037_054.jsonl
  episodes_eval_E3-steer-only_055_072.jsonl
  episodes

In [50]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   merge the shard logs
# READS   data/<RUN_DIR>/episodes_eval_*.jsonl
# WRITES  data/<RUN_DIR>/episodes.jsonl
# TIME    ~1 min
# SKIP?   no
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh eval --merge

+ python3 scripts/consolidate_evals.py
93 shard log(s):
  episodes_eval_E1_001_018.jsonl
  episodes_eval_E1_019_036.jsonl
  episodes_eval_E1_037_054.jsonl
  episodes_eval_E1_055_072.jsonl
  episodes_eval_E1_073_089.jsonl
  episodes_eval_E1_090_106.jsonl
  episodes_eval_E2_001_018.jsonl
  episodes_eval_E2_019_036.jsonl
  episodes_eval_E2_037_054.jsonl
  episodes_eval_E2_055_072.jsonl
  episodes_eval_E2_073_089.jsonl
  episodes_eval_E2_090_106.jsonl
  episodes_eval_E3-guard-only_001_001.jsonl
  episodes_eval_E3-guard-only_001_018.jsonl
  episodes_eval_E3-guard-only_019_036.jsonl
  episodes_eval_E3-guard-only_037_054.jsonl
  episodes_eval_E3-guard-only_055_072.jsonl
  episodes_eval_E3-guard-only_073_089.jsonl
  episodes_eval_E3-guard-only_090_106.jsonl
  episodes_eval_E3-steer-only_001_001.jsonl
  episodes_eval_E3-steer-only_001_018.jsonl
  episodes_eval_E3-steer-only_019_036.jsonl
  episodes_eval_E3-steer-only_037_054.jsonl
  episodes_eval_E3-steer-only_055_072.jsonl
  episodes_eval_E3-s

---

## Stage: analysis — no model calls, all `$0`

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   analyse - freeze, analyse, fit, figures, consistency
# READS   data/<RUN_DIR>/episodes.jsonl
# WRITES  data/<RUN_DIR>/analysis.json, results_real.json, figures/
# TIME    ~2 min
# SKIP?   no
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/pipeline.sh analyse

+ python3 scripts/freeze_results.py --experiment main
expected 1590 cells, have 1590 of them (1537 extra cells present too, e.g. from other experiments)

wrote /content/ceg-mem/data/results_real.json (frozen, 3217 episodes)
+ python3 scripts/analyze.py

=== oracle_calls_to_accept ===
  [dead    ] 
  [hard    ] 
  [medium  ] 
  [easy    ] 
  [too_easy] 
  [overall ] no_memory=5.575925925925926  untyped=2.039094650205761  typed=3.4020975056689347

=== redundant_attempts ===
  [dead    ] 
  [hard    ] 
  [medium  ] 
  [easy    ] 
  [too_easy] 
  [overall ] no_memory=4.816981132075472  untyped=8.120754716981132  typed=5.124528301886793  transcript=4.0

=== success_at_b ===
  [dead    ] 
  [hard    ] 
  [medium  ] 
  [easy    ] 
  [too_easy] 
  [overall ] no_memory=0.6698113207547169  untyped=0.6698113207547169  typed=0.6754716981132075  transcript=0.0

=== guard_evaluations ===
  [dead    ] 
  [hard    ] 
  [medium  ] 
  [easy    ] 
  [too_easy] 
  [overall ] no_memory=0.0  untyped=10.5471

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   freeze the ablation and sweep result sets
# READS   data/<RUN_DIR>/episodes.jsonl
# WRITES  data/<RUN_DIR>/results_*.json
# TIME    ~1 min
# SKIP?   no if you ran E3-E5
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
# the sub-grids, each frozen against the list it actually ran over
!python3 scripts/freeze_results.py --experiment ablation
!python3 scripts/freeze_results.py --experiment oracle_sweep --sweep-programs-from {RUN_DATA}/sweep_programs.txt
!python3 scripts/freeze_results.py --experiment typing_sweep --sweep-programs-from {RUN_DATA}/sweep_programs.txt

### The three post-hoc scripts this branch adds

No model calls, so no dollars — but two of them are hours of sandbox, and one is
the second-biggest compute item in the whole study after the grid itself:

| | sandbox runs | wall clock |
|---|---|---|
| `--check-regression` (paid during E2, not here) | 4,200 for the F2P/P2P split, once per program, plus one full pass per accepted patch ≈ **31k** | 1.3–3.4 h |
| `measure_redundancy.py` · `measure_patch_quality.py` | none — they only read logs | minutes |
| **`measure_typing_coherence.py`** | 8,690 patches × min(60, pool size) ≈ **327k** | **14–36 h** |

Pilot the coherence script on 20 tasks first (~62k runs, ~4 h), read the `caps`
block it writes, and only then decide whether the full run earns its day.

And read `c_hat` against the two numbers now printed beside it. On its own it is
uninterpretable: a θ that shatters every patch into its own class scores a
perfect homogeneity of 1.000 **and** a random-baseline of 1.000 — lift zero. The
`competing` block scores the obvious alternatives (bucket by the name of the
refuting case, or by θ's `property` half alone). If θ does not beat both, the
lattice is not carrying its keep.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   post-hoc: redundancy and patch quality
# READS   data/<RUN_DIR>/episodes.jsonl
# WRITES  data/<RUN_DIR>/redundancy.json, patch_quality.json
# TIME    ~5 min
# SKIP?   no
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!python3 scripts/measure_redundancy.py        # #2,3,4,5,7,8,11,18,33 + pass@k (the repeated-sampling baseline)
!python3 scripts/measure_patch_quality.py     # #21 correct/plausible, #23 verbosity, #22 regression rate

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   post-hoc: typing coherence, 20-task pilot
# READS   data/<RUN_DIR>/episodes.jsonl
# WRITES  data/<RUN_DIR>/typing_coherence.json
# TIME    ~20 min
# SKIP?   no - run the pilot before the full one and read its caps
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!python3 scripts/measure_typing_coherence.py --limit-tasks 20   # pilot first, read the caps

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   post-hoc: typing coherence, full
# READS   data/<RUN_DIR>/episodes.jsonl
# WRITES  data/<RUN_DIR>/typing_coherence.json
# TIME    hours
# SKIP?   yes if the pilot is enough
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
!nohup python3 scripts/measure_typing_coherence.py > {RUN_LOGS}/coherence.log 2>&1 &
print("launched; tail {RUN_LOGS}/coherence.log with the cell below")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   watch the coherence run
# READS   logs/<RUN_DIR>/coherence.log
# WRITES  nothing
# TIME    instant
# SKIP?   re-run as often as you like
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
!pgrep -fa measure_typing_coherence.py | head -1 || echo "not running"
!tail -5 {RUN_LOGS}/coherence.log

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   assert the pre-registered claims
# READS   data/<RUN_DIR>/analysis.json
# WRITES  nothing
# TIME    ~10 s
# SKIP?   NO. This is the falsifier check
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# Run last, and run it again before submitting.
!python3 scripts/check_consistency.py

### `measure_coherence.py` — Definition 3.1's coherence, measured

`measure_typing_coherence.py` above is the θ-labelling audit. This is the other
one: for every (task, failure-type) bucket holding two or more refuted
attempts, it re-executes one attempt's patch against the other's
counterexample. The ratio is the coherence Proposition 4.5's O(1) guard is only
sound at.

It is hours of sandbox time, so run `--plan-only` first and read the estimate.
Since 2026-09-01 it prints a progress line every `--progress-every` seconds and
**checkpoints the report on every one of them** — a run killed at hour three
leaves a valid `coherence_report.json` with `"partial": true` and the pair
counts, instead of leaving nothing at all.

In [ ]:
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
# Cost first. Nothing is executed by --plan-only.
!python3 scripts/measure_coherence.py --plan-only

In [ ]:
import os; os.chdir(WORKDIR)
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
os.makedirs(RUN_LOGS, exist_ok=True)
# Background, like every other long stage. --progress-every governs BOTH how
# often it prints and how much work a kill can cost, because the checkpoint
# rides the same clock.
!nohup python3 scripts/measure_coherence.py --progress-every 60 \
    > {RUN_LOGS}/coherence_measure.log 2>&1 &
print("launched; watch it with the next cell")

In [ ]:
import os; os.chdir(WORKDIR)
RUN_DATA = f"data/{RUN_DIR}" if RUN_DIR else "data"
RUN_LOGS = f"logs/{RUN_DIR}" if RUN_DIR else "logs"
!pgrep -fa measure_coherence.py | head -1 || echo "not running"
!tail -4 {RUN_LOGS}/coherence_measure.log

import json, pathlib
p = pathlib.Path(f"{RUN_DATA}/coherence_report.json")
if p.exists():
    b = json.loads(p.read_text())
    print("\npartial:", b.get("partial"))
    for g, v in b["granularities"].items():
        part = v.get("partial")
        where = f"  ({part['pairs_done']}/{part['pairs_total']} pairs)" if part else "  complete"
        print(f"  {g:8s} coherence = {v['cross_refutation_rate']['pooled']}{where}")

---

## 13. Optional: a second proposer on the sweep subset

**Do not replace qwen.** π is a property of the model: the 526 programs were
screened under `qwen2.5-coder:7b` and the corpus was banded on those numbers, so
switching the proposer invalidates the screen and forces it to be redone. Worse,
a stronger proposer shifts the π̂ distribution *right* — draining `dead` and
`hard`, which are exactly the two bands where three of the four theoretical
results are most visible. A stronger model can make the effect **harder** to see.

What is worth doing is *adding* a second proposer on the 30-task sweep subset.
It patches `DESIGN.md`'s own open item ("a single small proposer... ideally
checked against a stronger model on a subset") with a number instead of a
sentence, and it makes `#14 cost-of-pass` real money rather than a repricing.
`freeze_results.py` already refuses to mix two models in one freeze and `model`
is in the cell key, so there is no way for it to contaminate the main grid.

The cell refuses to run unless `CLOUD_API_KEY` is set in §1.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   optional: the cloud proposer, sweep subset only
# READS   data/<RUN_DIR>/sweep_programs.txt
# WRITES  data/<RUN_DIR>/episodes_eval_*.jsonl
# TIME    ~1 h, costs real money
# SKIP?   yes - it is a generality check, never the reported grid
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
if not CLOUD_API_KEY:
    print("CLOUD_API_KEY is empty in §1 - skipping the cloud arm (this is the default)")
else:
    env = (f"LLM_API_KEY={CLOUD_API_KEY} "
           f"PRICE_IN_PER_MTOK={CLOUD_PRICE_IN} PRICE_OUT_PER_MTOK={CLOUD_PRICE_OUT} "
           f"BUDGET_USD_CAP={CLOUD_BUDGET} CONTEXT_LENGTH={CLOUD_CONTEXT}")
    # BUDGET_USD_CAP here is the TOTAL: fleet.sh divides it by --shards, because
    # src.llm.spent() only ever sees the ledger of its own process.
    print(f"{env}\n")
    !{env} bash scripts/fleet.sh eval --exp E1 --shards 4 -- --backend cloud --model {CLOUD_MODEL}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   fleet status
# READS   logs/<RUN_DIR>/fleet/
# WRITES  nothing
# TIME    instant
# SKIP?   re-run as often as you like
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/fleet.sh status

---

# After a disconnect

Ollama does not survive a stopped runtime, and neither does `/content`. Drive
does, and so does everything in it. To carry on: run **§1** (config), **§3**
(mount), **§4** (clone), **§5** (links), **§6** (deps), **§8** (test data),
**§9** (Ollama), **§10** (`.env`), **§11** (verify) — then re-run the stage cell
you were on with the identical arguments.

The re-run is not wasted. Every model call already bought replays from the cache
on Drive; only the oracle, which is not cached, executes again.

The cell below is those steps collapsed into one, for a session that only needs
to resume.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   resume after a runtime recycle
# READS   your Drive
# WRITES  re-establishes the symlinks and the server
# TIME    ~5 min
# SKIP?   NO after a disconnect; skip on a fresh run
# ────────────────────────────────────────────────────────────────────────────
from google.colab import drive; drive.mount('/content/drive')
import os; os.chdir(WORKDIR)
!git pull --ff-only
!pip install -q -r requirements.txt && apt-get -qq install -y lsof > /dev/null

import subprocess, os, time, urllib.request
env = dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{OLLAMA_PORT}",
           OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH),
           OLLAMA_NUM_PARALLEL="1", OLLAMA_KEEP_ALIVE="60m")
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("logs/ollama.log", "a"), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2); break
    except Exception: time.sleep(1)

!bash scripts/serve_local.sh --port {OLLAMA_PORT}
!bash scripts/pipeline.sh

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   fleet status after a resume
# READS   logs/<RUN_DIR>/fleet/
# WRITES  nothing
# TIME    instant
# SKIP?   yes
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# Where the last fleet got to. The manifest survives the runtime because logs/
# is on Drive; the PROCESSES do not, so shards from a dead session read as
# KILLED. Re-launch the identical fleet - every call already bought replays free.
!bash scripts/fleet.sh status

---

# When something refuses

**`served context is 4096, not 32768`** — the GPU did not have room for the
window Ollama was asked for, or a server was already listening on the port with a
different one. Stop whatever is on the port, then re-run §9. Do not work around
it: the numbers would be wrong in a way nothing downstream can detect.

**`data/tasks.json missing` / `stage X reads Y`** — a stage was skipped.
`bash scripts/pipeline.sh` names the one to run.

**Every cell re-runs instead of skipping** — something in the experiment cell key
moved, and in practice it is `model`. Check the `model` field in the episode log
before assuming the resume logic is broken.

**`BudgetExceeded`** — local calls are priced at zero, so this cannot fire on a
healthy run. It is a tripwire: something has repointed the client at a paid
endpoint. Find out what rather than raising the cap.

**Drive quota, or `Transport endpoint is not connected`** — the mount dropped
mid-run. Re-run §3 and the stage cell; nothing is corrupted, because rounds are
appended one at a time.

`RUNBOOK.md` §9 is the full list, including the failure modes that are not
Colab-specific.

**`fleet.sh: a fleet is still running`** — one fleet at a time, by design: two
manifests interleaved is not a record of anything. `fleet.sh status`, then `stop`
or `wait`.

**A shard reads `KILLED`** — no pid and no recorded exit status, which is what a
dead Colab runtime leaves behind. Distinct from `exit N`, which is something the
shard found rather than something that happened to it. Re-launch the identical
fleet.

**Every shard reads `exit 2` immediately** — read one log (`fleet.sh tail 1`).
Almost always the server: `ollama not on PATH`, or nothing listening on
`OLLAMA_PORT` because §9 was skipped after a reconnect.

**`--backend cloud needs BUDGET_USD_CAP`** — export the TOTAL for the fleet and
let `fleet.sh` divide it. There is no safe default: the cap is per-process and
every shard has its own ledger.

**The screen ran but every band is empty except `dead`** — `--calls` was omitted,
so the shards took `screen_shard.sh`'s default depth instead of the study's.
π̂ lives on a grid of `1/K`. `fleet.sh` warns about this at launch.